Practical Exam: House sales
RealAgents is a real estate company that focuses on selling houses.

RealAgents sells a variety of types of house in one metropolitan area.

Some houses sell slowly and sometimes require lowering the price in order to find a buyer.

In order to stay competitive, RealAgents would like to optimize the listing prices of the houses it is trying to sell.

They want to do this by predicting the sale price of a house given its characteristics.

If they can predict the sale price in advance, they can decrease the time to sale.

Data
The dataset contains records of previous houses sold in the area.

Column Name	Criteria
house_id	Nominal. Unique identifier for houses. Missing values not possible.
city	Nominal. The city in which the house is located. One of 'Silvertown', 'Riverford', 'Teasdale' and 'Poppleton'. Replace missing values with "Unknown".
sale_price	Discrete. The sale price of the house in whole dollars. Values can be any positive number greater than or equal to zero.Remove missing entries.
sale_date	Discrete. The date of the last sale of the house. Replace missing values with 2023-01-01.
months_listed	Continuous. The number of months the house was listed on the market prior to its last sale, rounded to one decimal place. Replace missing values with mean number of months listed, to one decimal place.
bedrooms	Discrete. The number of bedrooms in the house. Any positive values greater than or equal to zero. Replace missing values with the mean number of bedrooms, rounded to the nearest integer.
house_type	Ordinal. One of "Terraced" (two shared walls), "Semi-detached" (one shared wall), or "Detached" (no shared walls). Replace missing values with the most common house type.
area	Continuous. The area of the house in square meters, rounded to one decimal place. Replace missing values with the mean, to one decimal place.
Task 1
The team at RealAgents knows that the city that a property is located in makes a difference to the sale price.

Unfortuntately they believe that this isn't always recorded in the data.

Calculate the number of missing values of the city.

You should use the data in the file "house_sales.csv".

Your output should be an object missing_city, that contains the number of missing values in this column.

In [ ]:
import pandas as pd
import numpy as np

house = pd.read_csv("house_sales.csv")

missing_city_df = house[house['city'] == '--']
missing_city = missing_city_df.city.count()
missing_city

Task 2
Before you fit any models, you will need to make sure the data is clean.

The table below shows what the data should look like.

Create a cleaned version of the dataframe.

You should start with the data in the file "house_sales.csv".

Your output should be a dataframe named clean_data.

All column names and values should match the table below.

Column Name	Criteria
house_id	Nominal. Unique identifier for houses. Missing values not possible.
city	Nominal. The city in which the house is located. One of 'Silvertown', 'Riverford', 'Teasdale' and 'Poppleton' Replace missing values with "Unknown".
sale_price	Discrete. The sale price of the house in whole dollars. Values can be any positive number greater than or equal to zero.Remove missing entries.
sale_date	Discrete. The date of the last sale of the house. Replace missing values with 2023-01-01.
months_listed	Continuous. The number of months the house was listed on the market prior to its last sale, rounded to one decimal place. Replace missing values with mean number of months listed, to one decimal place.
bedrooms	Discrete. The number of bedrooms in the house. Any positive values greater than or equal to zero. Replace missing values with the mean number of bedrooms, rounded to the nearest integer.
house_type	Ordinal. One of "Terraced", "Semi-detached", or "Detached". Replace missing values with the most common house type.
area	Continuous. The area of the house in square meters, rounded to one decimal place. Replace missing values with the mean, to one decimal place.

In [ ]:
import pandas as pd

df = pd.read_csv("house_sales.csv" , na_values=["--"])

df['city'].fillna('Unknown', inplace=True)

df.dropna(subset=['sale_price'], inplace=True)

df['sale_date'].fillna('2023-01-01', inplace=True)

df['months_listed'].fillna(df['months_listed'].mean().round(1), inplace=True)

df['bedrooms'].fillna(round(df['bedrooms'].mean()), inplace=True)

df['house_type'].replace({'Det.' : 'Detached' ,'Terr.' : 'Terraced' , 'Semi' :  'Semi-detached'} , inplace=True)

df['area'] = df['area'].str.replace(' sq.m.', '').astype(float)

df['area'].fillna(df['area'].mean(), inplace=True)

clean_data = df.copy()
clean_data

Task 3
The team at RealAgents have told you that they have always believed that the number of bedrooms is the biggest driver of house price.

Producing a table showing the difference in the average sale price by number of bedrooms along with the variance to investigate this question for the team.

You should start with the data in the file 'house_sales.csv'.

Your output should be a data frame named price_by_rooms.

It should include the three columns bedrooms, avg_price, var_price.

Your answers should be rounded to 1 decimal place.

In [ ]:
price_by_rooms = clean_data.groupby('bedrooms')['sale_price'].agg(['mean', 'var'])
price_by_rooms.columns = ['avg_price', 'var_price']
price_by_rooms = price_by_rooms.round(1)
price_by_rooms.reset_index(inplace=True)
price_by_rooms

Task 4
Fit a baseline model to predict the sale price of a house.

Fit your model using the data contained in “train.csv”

Use “validation.csv” to predict new values based on your model. You must return a dataframe named base_result, that includes house_id and price. The price column must be your predicted values.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

train = pd.read_csv("train.csv")
validate = pd.read_csv("validation.csv")

train = train.drop(['sale_date'],axis=1)

label_encoder_house = LabelEncoder()
label_encoder_city = LabelEncoder()
train['house_type'] = label_encoder_house.fit_transform(train['house_type'])
train['city'] = label_encoder_city.fit_transform(train['city'])

X = train.drop(['sale_price'], axis=1)
y = train.sale_price

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=40)

preprocessor = ColumnTransformer(
    transformers=[
        ('num' , 'passthrough',
         X.columns)
    ])

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=40))
])

model.fit(X_train, y_train)
prediction = model.predict(X_test)

mse = mean_squared_error(y_test, prediction)
rmse = np.sqrt(mse).round(1)
print("RMSE:", rmse)

validate2 = validate.copy()
validate2 = validate2.drop(['sale_date'],axis=1)

validate2['house_type'] = label_encoder_house.transform(validate2['house_type'])
validate2['city'] = label_encoder_city.transform(validate2['city'])

base_result = validate2.copy()
base_result['price'] = model.predict(base_result)
base_result

Task 5
Fit a comparison model to predict the sale price of a house.

Fit your model using the data contained in “train.csv”

Use “validation.csv” to predict new values based on your model. You must return a dataframe named compare_result, that includes house_id and price. The price column must be your predicted values.

In [ ]:
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

X = X
y = y

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=40)

comparison_model = LinearRegression()
comparison_model.fit(X_train, y_train)
comparison_predictions = comparison_model.predict(X_test)

comparison_mse = mean_squared_error(y_test, comparison_predictions)
comparison_rmse = np.sqrt(comparison_mse).round(1)
print("RMSE:", comparison_rmse)

compare_result = validate2.copy()
compare_result['price'] = comparison_model.predict(compare_result)
compare_result